# Kitsune IoT Attack Trainer

Binary benign-versus-attack model trained from the supplied Kitsune feature and label CSV files.

In [ ]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
MAX_ROWS_PER_CLASS_PER_DATASET = 25_000
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/kitsune_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
all_csv = sorted(INPUT_ROOT.rglob('*.csv'))
label_files = [path for path in all_csv if 'label' in path.name.lower()]
feature_files = [path for path in all_csv if path not in label_files]
print(f'Feature files: {len(feature_files)}')
print(f'Label files: {len(label_files)}')

def clean_name(path):
    return re.sub(r'[^a-z0-9]+', '', re.sub(r'labels?|dataset|features?', '', path.stem.lower()))

def match_label(feature_path):
    nearby = [path for path in label_files if path.parent == feature_path.parent] or label_files
    return max(nearby, key=lambda path: len(set(clean_name(feature_path)) & set(clean_name(path))))

def load_binary_labels(path):
    table = pd.read_csv(path, header=None, low_memory=False)
    possible = []
    for column in table.columns:
        values = pd.to_numeric(table[column], errors='coerce')
        unique = set(values.dropna().unique())
        if 2 <= len(unique) <= 10:
            possible.append(values)
    if not possible:
        raise ValueError(f'No binary label vector found in {path.name}')
    return possible[-1].fillna(0).ne(0).astype('int8')

def load_class_block(feature_path, labels, start, class_id, size):
    block = pd.read_csv(feature_path, header=None, skiprows=start, nrows=size)
    block.columns = [f'feature_{index}' for index in range(block.shape[1])]
    block_labels = labels.iloc[start:start + len(block)].reset_index(drop=True)
    block = block.loc[block_labels.eq(class_id).to_numpy()].copy()
    block['Label'] = class_id
    return block

frames = []
for feature_path in feature_files:
    labels = load_binary_labels(match_label(feature_path))
    benign_positions = np.flatnonzero(labels.eq(0).to_numpy())
    attack_positions = np.flatnonzero(labels.eq(1).to_numpy())
    count = min(MAX_ROWS_PER_CLASS_PER_DATASET, len(benign_positions), len(attack_positions))
    if count == 0:
        print(f'Skipping {feature_path.name}: one class is unavailable')
        continue
    benign = load_class_block(feature_path, labels, int(benign_positions[0]), 0, count)
    attack = load_class_block(feature_path, labels, int(attack_positions[0]), 1, count)
    frame = pd.concat([benign, attack], ignore_index=True)
    frame['source_dataset'] = feature_path.stem
    frames.append(frame)
    print(f'Loaded {len(benign):,} benign + {len(attack):,} attack rows: {feature_path.name}')

if not frames:
    raise ValueError('No usable Kitsune feature/label pairs found.')
dataset = pd.concat(frames, ignore_index=True, sort=False)
if 'Label' not in dataset or 'source_dataset' not in dataset:
    raise RuntimeError('Feature/label column construction failed.')
print(dataset['Label'].value_counts().rename(index={0: 'benign', 1: 'attack'}))

In [ ]:
feature_columns = [column for column in dataset.columns if column.startswith('feature_')]
X = dataset[feature_columns].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
y = dataset['Label'].astype('int8')
if set(y.unique()) != {0, 1}:
    raise ValueError(f'Expected binary labels {{0, 1}}, got {sorted(y.unique())}')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
model = RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = {'rows': int(len(dataset)), 'feature_count': len(feature_columns), 'macro_f1': float(f1_score(y_test, predictions, average='macro')), 'class_counts': {'benign': int((y == 0).sum()), 'attack': int((y == 1).sum())}}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, predictions, labels=[0, 1], target_names=['benign', 'attack'], zero_division=0))
pd.DataFrame(confusion_matrix(y_test, predictions, labels=[0, 1]), index=['benign', 'attack'], columns=['benign', 'attack'])

In [ ]:
dataset.to_csv(OUTPUT_DIR / 'kitsune_training_sample.csv', index=False)
joblib.dump(model, OUTPUT_DIR / 'kitsune_random_forest.joblib')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame({'feature': feature_columns, 'importance': model.feature_importances_}).sort_values('importance', ascending=False).to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
print(f'Saved model and evaluation artifacts to {OUTPUT_DIR}')